# Linear Regression

**Week 3 · Lesson 2** &nbsp;·&nbsp; GBUS 564: Predictive Analytics &nbsp;·&nbsp; Fall 2026

<a target="_blank" href="https://colab.research.google.com/github/avorybryant/umw_gbus564_fall26/blob/main/notebooks/week3/lectures/02_linear_regression.ipynb">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

*Adapted from ISLP Ch. 3.6 Lab where relevant.*

In [ ]:
# Environment setup — identical across every notebook in the course.
# Installs the course package (dataset loaders + helpers) from the public
# course repo; Colab already has numpy/pandas/scikit-learn, so it's quick.
%pip install -q matplotlib git+https://github.com/avorybryant/umw_gbus564_fall26.git

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn import set_config

set_config(display="diagram")
plt.rcParams["figure.figsize"] = (8, 5)

RANDOM_SEED = 564  # GBUS 564 — reuse this seed everywhere for reproducibility
rng = np.random.default_rng(RANDOM_SEED)

## 1. Business Problem

**Business question:** Which characteristics of an individual home move its
sale price, and by roughly how much — so an agent or appraiser can point
sellers and buyers at the factors that actually matter, not just gut feel?

**Why it matters:** Pricing off a single number (like square footage alone)
leaves money on the table, and "the model says so" isn't an answer a client
will accept — they need the driver named and its size quantified in dollars.

## 2. Data

Sale prices for ~2,930 individual homes sold in Ames, Iowa (2006-2010),
compiled from the city assessor's public records by Dean De Cock as a
teaching-friendly alternative to the classic Boston Housing dataset. Every
effect below reads as **true of this house** — a real sale, not a
neighborhood average.

In [ ]:
from gbus564_coursework.utils.course_data import load_housing

df = load_housing()
df.head()

The Ames file has about 80 columns; this lesson works with a hand-picked
eight. They were chosen for four reasons:

- **Numeric and ready to use** — no categorical encoding needed, so a plain
  `LinearRegression` can fit them directly. (Homes with no basement or
  garage already come through as 0 from `load_housing()`.)
- **Characteristics of the house itself** — size, rooms, quality, age,
  basement, garage, lot, and baths are the factors an agent would actually
  name to a client, which is exactly the business question.
- **One overlapping pair, on purpose** — `TotRmsAbvGrd` largely repeats
  `GrLivArea` (bigger homes have more rooms). It's kept in so the
  correlation problem shows up below instead of staying hypothetical.
- **Few enough to read** — eight coefficients fit in one table and one
  chart, so every number can be interpreted.

Left out for now: categorical features like `Neighborhood` and
`SaleCondition`, which need encoding first (Section 4 shows what skipping
`SaleCondition` costs). This list is a judgment call, not a data-driven
selection — letting the data choose which features matter is the job of
ridge and lasso in Lessons 7-9.

In [ ]:
predictors = ["GrLivArea", "TotRmsAbvGrd", "OverallQual", "YearBuilt",
              "TotalBsmtSF", "GarageCars", "LotArea", "FullBath"]
df[predictors + ["SalePrice"]].describe()

### Data Prep

One check before fitting: **predictor correlation**. A quick screen for the
kind of relationship that makes "holding everything else fixed" hard to
take literally — the concern Lesson 1 flagged, and the one Lesson 6 breaks
on purpose.

In [ ]:
df[predictors].corr().round(2)

`GrLivArea` (above-grade living area) and `TotRmsAbvGrd` (total rooms
above grade) move together (ρ ≈ 0.81) — bigger homes have more rooms,
unsurprisingly. Keep that in mind once we get to individual coefficients:
"holding square footage fixed" while room count changes is a scenario that
barely exists in this data.

**Hold out the test set.** The last step before any model sees the data:
`train_test_split` sets aside 20% of homes the model never trains on, so
Section 4 measures how it prices *new* sales, not how well it memorized
old ones. Nothing here needs encoding or scaling, so there's no
preprocessing to define — Section 3 fits straight onto the training rows.

In [ ]:
from sklearn.model_selection import train_test_split

X, y = df[predictors], df["SalePrice"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED
)

print(f"Training homes: {len(X_train):,}   Test homes: {len(X_test):,}")

## 3. Model

Baseline first — the reference the model must beat — then linear
regression on all eight home-level predictors.

In [ ]:
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression

baseline = DummyRegressor(strategy="mean").fit(X_train, y_train)
model = LinearRegression().fit(X_train, y_train)

### Least Squares Has a Closed Form

`LinearRegression().fit()` doesn't search for the best line — minimizing
RSS has an exact answer you can write down. Stack the predictors into a
matrix $X$ (with a leading column of 1s for the intercept) and the prices
into a vector $y$. The model is the same straight line from Lesson 1,
written for every home at once:

$$y = X\beta + \varepsilon$$

Least squares picks the $\beta$ that makes the residuals as small as
possible — it minimizes

$$\text{RSS}(\beta) = \lVert y - X\beta \rVert^2 = \sum_i \big(y_i - x_i^\top \beta\big)^2$$

Setting the derivative of RSS to zero gives the **normal equations**,
which solve for $\beta$ directly:

$$\hat\beta = (X^\top X)^{-1} X^\top y$$

One line of linear algebra, no iterating. Solve it by hand on the same
training rows and compare against what scikit-learn fit:

In [ ]:
# Normal equations: β̂ = (XᵀX)⁻¹ Xᵀy — just matrix multiplication.
# (Production libraries avoid the explicit inverse for numerical stability;
# on a small, well-behaved X like this one it makes no difference.)
X_mat = np.column_stack([np.ones(len(X_train)), X_train.to_numpy()])
beta_hat = np.linalg.inv(X_mat.T @ X_mat) @ X_mat.T @ y_train.to_numpy()

comparison = pd.DataFrame(
    {
        "Closed form β̂": beta_hat,
        "scikit-learn": np.r_[model.intercept_, model.coef_],
    },
    index=pd.Index(["Intercept"] + predictors, name="Term"),
)

comparison.style.format({
    "Closed form β̂": "{:,.4f}",
    "scikit-learn": "{:,.4f}"
})

## 4. Evaluate

Score against the baseline, then check the residuals — a good RMSE doesn't
mean the straight-line assumption actually holds.

In [ ]:
from sklearn.metrics import mean_squared_error

for name, m in [("Baseline", baseline), ("Linear regression", model)]:
    pred = m.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    print(f"{name:<18} RMSE: ${rmse:>9,.0f}")

### Reading a Residual Plot (Sandbox Calibration)

Before diagnosing the real fit below, calibrate on synthetic data where the
ground truth is known. Fit the same simple linear regression to three
regimes of the same underlying truth — only the noise-generating process
changes — and see what a violation actually looks like when nothing about
the model itself does.

In [ ]:
from gbus564_coursework.utils.sandbox import make_residual_diagnostics_sandbox

fig, axes = plt.subplots(1, 3, figsize=(13, 4), sharey=True)

for ax, regime, title in zip(
    axes,
    ["clean", "heteroscedastic", "curved"],
    ["Clean — constant variance", "Heteroscedastic — fan shape", "Curved — missing term"],
):
    X_demo, y_demo = make_residual_diagnostics_sandbox(regime=regime, rng=rng)
    demo_model = LinearRegression().fit(X_demo, y_demo)
    demo_fitted = demo_model.predict(X_demo)
    demo_resid = y_demo - demo_fitted

    ax.scatter(demo_fitted, demo_resid, s=14, alpha=0.4, color="#0B3D91")
    ax.axhline(0, color="black", linestyle="--", linewidth=1)
    ax.set_title(title, fontsize=11)
    ax.set_xlabel("Fitted value")

axes[0].set_ylabel("Residual")
plt.tight_layout()

Same true slope, same sample size — only the violation changes:

- **Clean** — residuals scatter evenly above and below zero with constant
  spread and no pattern. This is the target: nothing systematic left for
  the model to explain.
- **Heteroscedastic** — the spread visibly widens as the fitted value
  grows, the textbook fan shape. Standard errors and confidence intervals
  assume constant variance, so this is a sign to distrust them, not just a
  cosmetic quirk.
- **Curved** — the true relationship has a quadratic term the line can't
  capture, so residuals bow into a U: over-predicting in the middle,
  under-predicting at the extremes. A pattern like this points at a
  missing term, not noise.

Keep the fan shape and the U in mind — the real residual plot below shows
one of them.

In [ ]:
# Residual-vs-fitted plot: look for a pattern, not just a low error.
# Adapted from ISLP Ch.3 Lab's diagnostic plot.
fitted = model.predict(X_test)
resid = y_test - fitted

fig, ax = plt.subplots()
ax.scatter(fitted, resid, s=14, alpha=0.4, color="#0B3D91")
ax.axhline(0, color="black", linestyle="--", linewidth=1)
ax.set_xlabel("Fitted value ($)")
ax.set_ylabel("Residual ($)")
ax.set_title("Residuals vs. fitted — look for a pattern, not just a low error")
plt.tight_layout()

Two things stand out, neither of them random noise:

- **The residual spread fans out** as the fitted price climbs — the model
  is noticeably less precise on expensive homes than on typical ones. This
  is the standard sign that the constant-variance assumption behind
  standard errors and confidence intervals is strained, not just a
  cosmetic issue.
- **A handful of homes miss badly**, including one over $350,000 off. That
  home (`GrLivArea` 4,676 sq ft, quality rating 10, built 2007) has every
  feature of a $500,000+ property but sold for only $184,750 — its
  `SaleCondition` is `Partial`, Ames' code for a sale recorded before
  construction was finished. The model has no way to know that from square
  footage and room counts alone.

The straight-line story is useful through the middle of the market but
strains at the high end and on unusual sales — keep that caveat for
Section 6.

## 5. Interpret

Coefficients as a dollar story, with the uncertainty that goes with them —
the same standard error / confidence interval / p-value reading from
Lesson 1, now on real data.

In [ ]:
# statsmodels gives the SE / CI / p-value table sklearn doesn't; sklearn
# handled the train/test workflow above, statsmodels is purely for inference.
%pip install -q statsmodels
import statsmodels.api as sm

X_train_c = sm.add_constant(X_train)
ols = sm.OLS(y_train, X_train_c).fit()

summary = pd.DataFrame({
    "coef ($)": ols.params,
    "std err": ols.bse,
    "t": ols.tvalues,
    "p-value": ols.pvalues,
}).join(ols.conf_int().rename(columns={0: "CI 2.5%", 1: "CI 97.5%"}))
summary.round(2)

In [ ]:
from IPython.display import Markdown

def p_text(p):
    return "p < 0.001" if p < 0.001 else f"p = {p:.2g}"

def dollars(x):
    # A backslash before $ keeps Jupyter from reading the dollar sign as LaTeX math.
    return f"{'−' if x < 0 else ''}\\${abs(x):,.0f}"

coef, pval = ols.params, ols.pvalues

Markdown(f"""
- **Living area** — each additional square foot is associated with
  **{dollars(coef['GrLivArea'])}** more in sale price, holding the other seven
  characteristics fixed ({p_text(pval['GrLivArea'])}).
- **Overall quality** — each additional point on the 1-10 scale is associated
  with **{dollars(coef['OverallQual'])}** more ({p_text(pval['OverallQual'])}),
  the single largest lever in raw dollar terms per unit.
- **Room count** — comes back at **{dollars(coef['TotRmsAbvGrd'])}**: negative,
  and significant ({p_text(pval['TotRmsAbvGrd'])}). More rooms, *lower*
  predicted price? That's the Lesson 1 interpretation trap, live:
  `GrLivArea` and `TotRmsAbvGrd` are correlated (ρ ≈ 0.81, from the check
  above), so the model splits their shared signal unevenly — `GrLivArea`
  absorbs almost all of the positive size effect and `TotRmsAbvGrd` is left
  with an implausible negative one. Neither coefficient is individually
  trustworthy; **read them as a pair**.
""")

In [ ]:
# Standardized coefficients answer "which factor matters most" — raw
# coefficients aren't comparable across features measured in different units
# (square feet vs. a 1-10 quality score vs. a room count).
from sklearn.preprocessing import StandardScaler

X_train_std = StandardScaler().fit_transform(X_train)
std_model = LinearRegression().fit(X_train_std, y_train)

std_effects = pd.Series(std_model.coef_, index=predictors).sort_values()
std_effects.plot.barh(title="Standardized effect on sale price")
plt.xlabel("Effect ($ of price per std. dev. of predictor)")
plt.tight_layout()

`OverallQual` and `GrLivArea` dominate once every predictor is placed on
the same scale — quality and size are doing most of the work. `TotRmsAbvGrd`
still shows up, but per the correlation check above, read it alongside
`GrLivArea`, not as an independent lever.

## 6. Recommend

**Recommendation:** Anchor pricing conversations on overall quality and
living area first — they carry the most weight and are the most reliably
estimated. Use room count as a supporting, *bundled* signal alongside
square footage (more space *and* more rooms together, not either
coefficient read in isolation).

**Caveats / next steps:** This is 2006-2010 sales data from one Iowa city,
so treat dollar amounts as specific to that market and period, not a
universal price rule. The residual plot in Section 4 already showed the
model losing precision at the high end and missing badly on at least one
unusual (`Partial`-condition) sale — a straight line is a useful
approximation, not a promise. `GrLivArea`/`TotRmsAbvGrd` correlation means
individual coefficients here are exactly the kind of estimate that gets
more (or less) trustworthy under ridge/lasso — Lesson 6 breaks this on
purpose, and Lessons 7-9 fix it.